# 07. Distance & Similarity Metrics## What is this notebook about?Many ML algorithms (KNN, K-Means, hierarchical clustering, recommender systems) need a way to measure **how similar** two data points are.This notebook introduces the most common distance/similarity metrics, when to use each, and how to compute them.## What you will learn1. **Euclidean distance** - the everyday "ruler" distance2. **Manhattan distance** - city-block distance3. **Cosine similarity** - angle between vectors (great for text)4. **Pearson correlation** - linear association between variables5. **Jaccard similarity** - overlap between sets## DatasetsIris (numerical features) and synthetic sets (for Jaccard).

In [ ]:
# If you're running locally, uncomment and run this once.# In Google Colab, all of these are pre-installed - you can skip this cell.# !pip install -q numpy pandas scikit-learn matplotlib seaborn scipy xgboost lightgbm

In [ ]:
# Standard imports used in every ML notebookimport numpy as np                 # numerical arraysimport pandas as pd                # tabular data (DataFrames)import matplotlib.pyplot as plt    # plottingimport seaborn as sns              # prettier statistical plots# Set up plot stylesns.set_style("whitegrid")plt.rcParams["figure.figsize"] = (10, 6)# Set a random seed so results are reproduciblenp.random.seed(42)

In [ ]:
from sklearn.datasets import load_irisfrom sklearn.metrics.pairwise import euclidean_distances, manhattan_distances, cosine_similarityfrom scipy.spatial.distance import pdist, squareform# Load Irisiris = load_iris(as_frame=True)X = iris.data.values     # convert to numpy arrayy = iris.target.valuesprint(f"Shape: {X.shape}")

## Step 1. Euclidean DistanceThis is the everyday "as the crow flies" distance. For two points (x1, y1) and (x2, y2):`distance = sqrt( (x1-x2)^2 + (y1-y2)^2 )`Generalizes to any number of dimensions.

In [ ]:
# Take the first 5 flowerssample = X[:5]# Pairwise euclidean distances - returns a 5x5 matrixprint("Euclidean distance matrix (5 flowers):")print(np.round(euclidean_distances(sample), 2))

**Reading the matrix:** entry [i, j] is the distance between flower i and flower j. Diagonal is 0 (same flower). Matrix is symmetric (distance from A to B = distance from B to A).## Step 2. Manhattan DistanceDistance you'd walk on a city grid - sum of absolute differences along each axis.`distance = |x1-x2| + |y1-y2|`

In [ ]:
print("Manhattan distance matrix (5 flowers):")print(np.round(manhattan_distances(sample), 2))

Notice Manhattan distances are larger than Euclidean (the diagonal shortcut is unavailable on a grid).## Step 3. Cosine SimilarityCosine similarity measures the **angle** between two vectors, **ignoring their magnitudes**. It ranges from -1 (opposite directions) to 1 (same direction), with 0 meaning perpendicular.This is the **go-to metric for text** because document length shouldn't affect similarity.

In [ ]:
print("Cosine similarity matrix (5 flowers):")print(np.round(cosine_similarity(sample), 3))

Almost all 1.0 here - because all flowers point in roughly the same direction in feature space (they all have positive values for all 4 measurements). Cosine works better when items can have very different magnitudes but similar profiles.## Step 4. Pearson CorrelationPearson measures **linear association** between two variables. Range from -1 to 1.- +1 = perfectly positively correlated- 0 = no linear relation- -1 = perfectly negatively correlated

In [ ]:
# Correlation between features (not between samples!)print("Pearson correlation between features:")print(iris.data.corr().round(2))# Visualize as a heatmapplt.figure(figsize=(6, 5))sns.heatmap(iris.data.corr(), annot=True, cmap="coolwarm", center=0)plt.title("Pearson Correlation between Iris Features")plt.show()

Petal length and petal width are very strongly correlated (0.96) - they grow together.## Step 5. Jaccard SimilarityFor comparing **sets**, not numbers. It's the size of the intersection divided by the size of the union.`Jaccard(A, B) = |A ∩ B| / |A ∪ B|`Used heavily in document deduplication, recommender systems, and shopping cart analysis.

In [ ]:
# Three users and the genres they likeuser1 = {"action", "comedy", "thriller", "drama"}user2 = {"comedy", "drama", "romance"}user3 = {"horror", "thriller", "action"}def jaccard_sim(a, b):    """Jaccard similarity between two sets."""    return len(a & b) / len(a | b)# Compare all pairsprint(f"User1 vs User2 Jaccard: {jaccard_sim(user1, user2):.2f}  (overlap: {user1 & user2})")print(f"User1 vs User3 Jaccard: {jaccard_sim(user1, user3):.2f}  (overlap: {user1 & user3})")print(f"User2 vs User3 Jaccard: {jaccard_sim(user2, user3):.2f}  (overlap: {user2 & user3})")

**A simple recommender:** find the user most similar to you (highest Jaccard), then recommend the genres they liked that you don't yet.## Step 6. Visualize the full Iris distance matrix

In [ ]:
# Pairwise distances between ALL 150 flowers, visualized as a heatmapdist = squareform(pdist(X, metric="euclidean"))plt.figure(figsize=(8, 6))sns.heatmap(dist, cmap="Greys")plt.title("Pairwise Euclidean Distance Matrix (150 flowers)")plt.xlabel("Flower index")plt.ylabel("Flower index")plt.show()

**The block pattern reveals the species!** Iris setosa (first 50 flowers) is far from the others. Versicolor (50-100) and virginica (100-150) are closer to each other but still distinct. This is exactly what clustering algorithms detect.## Step 7. Exercises1. **Find the most similar pair of flowers** with each metric (`np.argmin` of the off-diagonal entries).2. **Compare cosine vs euclidean** when one feature is on a much bigger scale:   ```python   X_unscaled = X.copy()   X_unscaled[:, 0] *= 1000   # blow up feature 0   ```   Euclidean breaks. Cosine still works.3. **Build a tiny recommender** that suggests songs to a user based on cosine similarity of listening histories.4. **Compute Hamming distance** between two equal-length binary strings.5. **Try the Jaccard metric** on Wikipedia article shingles to find duplicate articles.Next: **08 - K-Means Clustering**!